In [2]:
import os
from torchvision.io import read_image
import matplotlib.pyplot as plt
import pandas as pd
import torch
import numpy as np

os.environ['CHEAP_CACHE'] = 'cache'
device="cuda"

In [3]:
from cheap.pretrained import CHEAP_shorten_2_dim_128
pipeline = CHEAP_shorten_2_dim_128(return_pipeline=True)

100%|██████████████████████████████████████| 4.12k/4.12k [00:00<00:00, 2.81MB/s]
100%|██████████████████████████████████████| 4.12k/4.12k [00:00<00:00, 2.41MB/s]
100%|██████████████████████████████████████| 4.12k/4.12k [00:00<00:00, 2.07MB/s]
100%|██████████████████████████████████████| 4.12k/4.12k [00:00<00:00, 1.90MB/s]
Downloading: "https://huggingface.co/amyxlu/cheap-proteins/resolve/main/checkpoints/uhg29zk4/last.ckpt" to cache/checkpoints/uhg29zk4/last.ckpt


Using checkpoint at cache/checkpoints/uhg29zk4.


100%|██████████████████████████████████████| 1.09G/1.09G [00:20<00:00, 55.6MB/s]


Using tanh layer at bottleneck...
Finished loading HPCT model with shorten factor 2 and 128 channel dimensions.
Creating ESMFold embedding only model...
ESMFold embedding only model created in 72.98 seconds


In [18]:
sequences = [
    # >cath|current|12asA00/4-330
    "AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLL",
    # >cath|current|132lA00/2-129
    "VFGRCELAAAMRHGLDNYRGYSLGNWVCAAFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKIVSDGNGMNAWVAWRNRCGTDVQAWIRGCRL",
    # >cath|current|153lA00/1-185
    "RTDCYGNVNRIDTTGASCKTAKPEGLSYCGVSASKKIAERDLQAMDRYKTIIKKVGEKLCVEPAVIAGIISRESHAGKVLKNGWGDRGNGFGLMQVDKRSHKPQGTWNGEVHITQGTTILINFIKTIQKKFPSWTKDQQLKGGISAYNAGAGNVRSYARMDIGTTHDDYANDVVARAQYYKQHGY",
]

emb, mask = pipeline(sequences)
emb.shape, mask.shape

(torch.Size([3, 164, 128]), torch.Size([3, 164]))

In [10]:
from cheap.proteins import LatentToSequence

latent_to_sequence = LatentToSequence()
latent_to_sequence = latent_to_sequence.to("cuda")

In [7]:
emb.device

device(type='cuda', index=0)

In [8]:
uncompressed = pipeline.decode(emb, mask)
uncompressed.shape

torch.Size([3, 328, 1024])

In [17]:
seq = latent_to_sequence.to_sequence(uncompressed)
seq[-1]

['AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLLA',
 'VFGRCELAAAMRHGLDNYRGYSLGNWVCAAFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKIVSDGNGMNAWVAWRNRCGTDVQAWIRGCRLAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA',
 'RTDCYGNVNRIDTTGASCKTAKPEGLSYCGVSASKKIAERDLQAMDRYKTIIKKVGEKLCVEPAVIAGIISRESHAGKVLKNGWGDRGNGFGLMQVDKRSHKPQGTWNGEVHITQGTTILINFIKTIQKKFPSWTKDQQLKGGISAYNAGAGNVRSYARMDIGTTHDDYANDVVARAQYYKQHGYAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA']

In [25]:
metadata_path = "/home/wero/counterfactual-proteomics/datasets/tape/metadata.csv"
metadata = pd.read_csv(metadata_path)

In [49]:
def hamming_distance(str1: str, str2: str) -> int:
    """Calculate the Hamming distance between two strings.

    Args:
        str1: First string.
        str2: Second string.

    Returns:
        The Hamming distance between the two strings.
    """
    # Ensure the strings are of the same length
    if len(str1) != len(str2):
        raise ValueError("Strings must be of the same length")

    # Calculate the Hamming distance
    distance = sum(1 for x, y in zip(str1, str2, strict=False) if x != y)
    return distance

In [60]:
seqs = metadata.loc[:3, "primary"].tolist()
embs, masks = pipeline(seqs)
uncompressed = pipeline.decode(embs, masks)
decoded = latent_to_sequence.to_sequence(uncompressed)[-1]
for i in range(len(seqs)):
    decoded[i] = decoded[i][:-1]
    print(decoded[i] == seqs[i])
    print(decoded[i])
    print(seqs[i])
    print()

True
SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHKIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDERYK
SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHKIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDERYK

True
SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGRLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK
SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGRLPVPWPTLVTTLSYGAQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK

True
SKGEELFTGVVPILVELDGDVNGHKFSVSGE

In [ ]:
import tempfile
import subprocess
import os
import pickle


def decode_in_subprocess(args, apptainer_image="cheap_container.sif"):
    """Decodes embeddings to sequences using a separate apptainer process to avoid
    dependecy conflicts.

    Args:
        args (dict): Arguments to pass to the function.
                        For 'decode_to_sequence_list', args should be a list of dicts,
                        each with "x" and "mask".
        apptainer_image (str): Path to the Apptainer image.

    Returns:
        Any: The result returned from the cheap function.
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        input_file = os.path.join(tmpdir, "input.pkl")
        output_file = os.path.join(tmpdir, "output.pkl")

        # Save args to input_file
        with open(input_file, "wb") as f:
            pickle.dump(
                {
                    "args": args,  # Args is now expected to be the list for batch processing
                    "output_file": output_file,
                },
                f,
            )

        # Path to the script to run inside the container
        # TODO: move cheap code to dependecies once import execution is fixed
        script_path = os.path.join("..", "cheap-proteins", "decode_embeddings.py")

        cmd = [
        "srun",
        "--pty",
        "--overlap",
        "--jobid",
        "$SLURM_JOB_ID",
        "'apptainer",  # Need to quote apptainer to avoid issues with slurm. Opening quote.
        "exec",
        "--nv",
        "--env",
        "MPLBACKEND=",  # Needed to make this work in jupyter notebooks
        "--bind",
        f"{tmpdir}:/tmp",
        apptainer_image,
        "python",
        script_path,
        input_file,
        "'"]

        cmd = " ".join(cmd)

        print("command:\n", cmd)
        result = subprocess.run(cmd, capture_output=True, text=True, check=True, stdout=sys.stdout, shell=True)
        if result.returncode != 0:
            print("Subprocess failed:", result.stderr)
            raise RuntimeError("CHEAP subprocess failed")

        # Read output from output_file
        with open(output_file, "rb") as f:
            return pickle.load(f)

def embedding2sequence(x_list):
    processed_embeddings = []
    masks = []
    seq_lens = []
    min_val, max_val = (-0.9, 0.9)

    for x_embedding in x_list:
        print("Embedding shape:", x_embedding.shape)
        # Remove channel dimension
        x = x_embedding
        mask = torch.zeros(x.shape[0], dtype=torch.bool).to("cuda")
        avg_row_vals = torch.mean(x, dim=1)
        mask[torch.abs(avg_row_vals - 0) > 1e-4] = True
        seq_len = mask.sum().item()
        seq_lens.append(seq_len)

        x_scaled = (x - min_val) / (max_val - min_val)
        processed_embeddings.append(x_scaled)
        masks.append(mask)

    input_args = {"x": processed_embeddings, "mask": masks}

    # Call the CHEAP apptainer subprocess with the list of processed embeddings
    decoded_sequences = decode_in_subprocess(input_args)

    trimmed_sequences = []
    for i, seq_str in enumerate(decoded_sequences):
        trimmed_seq = seq_str[: seq_lens[i]]
        print(f"Original decoded sequence {i}:", seq_str)
        print(f"Trimmed sequence {i} (len {seq_lens[i]}):", trimmed_seq)
        trimmed_sequences.append(trimmed_seq)

    return trimmed_sequences

In [35]:
sequences = [
    # >cath|current|12asA00/4-330
    "AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLL",
    # >cath|current|132lA00/2-129
    "VFGRCELAAAMRHGLDNYRGYSLGNWVCAAFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKIVSDGNGMNAWVAWRNRCGTDVQAWIRGCRL",
    # >cath|current|153lA00/1-185
    "RTDCYGNVNRIDTTGASCKTAKPEGLSYCGVSASKKIAERDLQAMDRYKTIIKKVGEKLCVEPAVIAGIISRESHAGKVLKNGWGDRGNGFGLMQVDKRSHKPQGTWNGEVHITQGTTILINFIKTIQKKFPSWTKDQQLKGGISAYNAGAGNVRSYARMDIGTTHDDYANDVVARAQYYKQHGY",
]

embs, masks = pipeline(sequences)
emb_list = []
for i in range(len(embs)):
    mask = masks[i]
    emb = embs[i]
    emb[~mask, :] = 0
    emb_list.append(emb)

embedding2sequence(emb_list)

Embedding shape: torch.Size([164, 128])
Embedding shape: torch.Size([164, 128])
Embedding shape: torch.Size([164, 128])
command:
 srun --pty --overlap --jobid $SLURM_JOB_ID 'apptainer exec --nv --env MPLBACKEND= --bind /tmp/tmpz5kow8xf:/tmp cheap_container.sif python ../cheap-proteins/decode_embeddings.py /tmp/tmpz5kow8xf/input.pkl '


FileNotFoundError: [Errno 2] No such file or directory: "srun --pty --overlap --jobid $SLURM_JOB_ID 'apptainer exec --nv --env MPLBACKEND= --bind /tmp/tmpz5kow8xf:/tmp cheap_container.sif python ../cheap-proteins/decode_embeddings.py /tmp/tmpz5kow8xf/input.pkl '"